# HCFA 1500 Form Data Extraction Pipeline

This notebook implements a pipeline to extract structured data from HCFA 1500 medical claim forms using OCR (Optical Character Recognition) and Large Language Models (LLMs).

## Workflow:
1. **Setup**: Install necessary OCR engines and Python libraries.
2. **Image Preprocessing**: Prepare the form image for better OCR results.
3. **OCR**: Extract raw text and layout information from the image using Tesseract.
4. **LLM Extraction**: Send the raw OCR text to an LLM (e.g., GPT-4o) to parse specific fields (Patient Name, Policy Number, Diagnosis Codes, etc.) into structured JSON.

In [1]:
# @title Install Dependencies

!sudo apt-get install tesseract-ocr
!pip install pytesseract pillow openai pandas python-dotenv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [9]:
# @title Import Libraries & Configure
import pytesseract
from PIL import Image, ImageEnhance, ImageFilter
import os
import json
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

print("Libraries imported and OpenAI configured securely.")


Libraries imported and OpenAI configured securely.


In [10]:
# @title Define OCR & Processing Functions

def preprocess_image(image_path):
    """
    Enhances an image for better OCR results.
    """
    try:
        img = Image.open(image_path)
        img = img.convert('L')
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(2.0)
        img = img.filter(ImageFilter.SHARPEN)
        return img
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

def extract_text_from_image(image):
    """
    Uses Tesseract to extract text from a PIL Image object.
    """
    try:
        custom_config = r'--oem 3 --psm 3'
        text = pytesseract.image_to_string(image, config=custom_config)
        return text
    except Exception as e:
        return f"Error during OCR: {e}"

def parse_hcfa_with_llm(ocr_text):
    """
    Sends OCR text to OpenAI to extract structured HCFA 1500 fields.
    """
    prompt = f"""
    You are an expert medical billing assistant. I have performed OCR on a HCFA 1500 (CMS 1500) form.
    The text may be messy or unstructured due to OCR errors. Your job is to extract the following fields accurately into JSON format.

    Target Fields:
    - insurance_type (Box 1)
    - insured_id (Box 1a)
    - patient_name (Box 2)
    - patient_birth_date (Box 3)
    - insured_name (Box 4)
    - patient_address (Box 5)
    - diagnosis_codes (Box 21 - list of codes like A, B, C...)
    - service_lines (Box 24 - array of objects with date_of_service, cpt_code, charges)
    - federal_tax_id (Box 25)
    - total_charge (Box 28)

    Here is the OCR Text:
    ---------------------
    {ocr_text}
    ---------------------

    Return ONLY valid JSON. Do not include markdown formatting.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts data from medical forms."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error calling LLM: {e}"


In [11]:
# @title Run on an Image

image_path = "HCFA 1500.png"

if os.path.exists(image_path):
    print(f"Processing {image_path}...")

    processed_img = preprocess_image(image_path)

    if processed_img:
        print("Running OCR...")
        raw_text = extract_text_from_image(processed_img)
        print(f"OCR Complete. Extracted {len(raw_text)} characters.")

        print("Sending to LLM for extraction...")
        structured_data = parse_hcfa_with_llm(raw_text)

        try:
            data_json = json.loads(structured_data)
            print(json.dumps(data_json, indent=2))
        except:
            print("Raw LLM Output (could not parse JSON):")
            print(structured_data)
else:
    print(f"File not found.")

Processing HCFA 1500.png...
Running OCR...
OCR Complete. Extracted 4424 characters.
Sending to LLM for extraction...
Raw LLM Output (could not parse JSON):
```json
{
  "insurance_type": "BlueCross BlueShield of Texas",
  "insured_id": "1234567890",
  "patient_name": "Powers, Billy H.",
  "patient_birth_date": "11/20/88",
  "insured_name": "Powers, Billy H.",
  "patient_address": "123 Super Power Lane, Austin, TX 78705",
  "diagnosis_codes": ["F50.2", "F41.4"],
  "service_lines": [
    {
      "date_of_service": {
        "from": "01/31/23",
        "to": "01/31/23"
      },
      "cpt_code": "90837",
      "charges": "185.00"
    }
  ],
  "federal_tax_id": "87-12345678",
  "total_charge": "185.00"
}
```
